# Parallel-route LM pilot: is retrieval's 0.84 a ceiling or a budget?

**Run all. Nothing to set.** Send back `GATE2_SUMMARY.txt`.

## The question

The earlier pilot nested the routes: retrieval read its queries, keys and values from the GRU
output `G`, so deleting or pricing recurrence took retrieval with it. Patched, both routes read
the trunk states `S` and meet only as a sum of logits:

```
S = trunk(x)            transformer trunk, a real sequence model
S -> GRU      -> h_rec  -> W_h  |
                                |-> logits = l_h + l_c
S -> q,k,v    -> c      -> W_c  |
```

After the patch, retrieval-only reached about 0.8367 at L=3 and so failed the 0.95 viability
screen, and Stage 2 was never run. But that run had a fixed 6000-step budget. A screen that stops
early measures the budget, not the route's capability, and the whole point of a viability screen
is to measure capability. So this notebook re-runs the screen at 4x the budget with the accuracy
curve logged throughout, and asks whether retrieval is still climbing or has plateaued.

## The decision rule, fixed before the run

Written to `gate2_rule.json` before any training starts:

- **VIABLE** — retrieval-only reaches 0.95 at any checkpoint. A receiver exists; the follow-up
  hybrid run then checks whether the free hybrid is retrieval-dominant, which is the precondition
  for steering.
- **CEILING** — it never reaches 0.95 and improves by less than 0.02 over the final third of
  training. The route is capacity-limited, and that is a measured boundary rather than an
  artifact of when training stopped. This is a reportable negative.
- **UNDERTRAINED** — it never reaches 0.95 but is still improving by more than 0.02. Neither
  conclusion is available and the honest report is that the budget was insufficient.

Only the first outcome leads anywhere. The other two are results, not failures, and neither is a
licence to keep extending the budget until the number moves — that would make the screen
post-hoc, which is exactly what the paper's prospective design avoids.

## What this is not

This is a screen, not a steering experiment. No price, no arms. If retrieval turns out viable and
the free hybrid is retrieval-dominant, the four-arm run is a separate session.

In [ ]:
import glob, hashlib, json, math, os, pathlib, random, subprocess, sys, time, traceback
from contextlib import nullcontext
from dataclasses import dataclass, asdict

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib as mpl
import matplotlib.pyplot as plt

PRESET = "full"           # "smoke" runs the whole pipeline in a few minutes to check it works

IN_KAGGLE = pathlib.Path("/kaggle/working").exists()
if IN_KAGGLE:
    WORKDIR = "/kaggle/working/lm_route_gate2"
    SEARCH_ROOTS = [pathlib.Path("/kaggle/input"), pathlib.Path("/kaggle/working")]
else:
    WORKDIR = os.environ.get("LM_WORKDIR", str(pathlib.Path.cwd() / "lm_route_gate2"))
    SEARCH_ROOTS = [pathlib.Path(WORKDIR)]


def P(full, smoke):
    return full if PRESET == "full" else smoke


# --- the question ---
GATE_LOAD    = 3                 # the load where retrieval reached 0.8367 in the earlier pilot
VIABLE_ACC   = 0.95              # the paper's route-sufficiency screen threshold
STILL_RISING = 0.02              # improvement over the final third that counts as "not plateaued"
LONG_STEPS   = P(24000, 150)     # 4x the earlier 6000-step budget
LOG          = P(1000, 50)
PREV_ACC     = 0.8367            # for reference in the summary only

# --- fixed from the earlier pilot so the comparison is like-for-like ---
SEEDS        = [0]
N_EVAL_REC   = P(8, 2)           # x 32 recall sequences
N_EVAL_LM    = P(16, 2)          # x 8 validation windows
N_STORIES    = P(120_000, 5_000)
LM_VOCAB     = 8000
SEED0, EVAL_SEED, FORMAT_VERSION = 0, 4321, 2     # v2: routes read S, not G
TIME_BUDGET_H = P(8.0, 0.3)
CKPT_MINUTES  = P(15.0, 2.0)


@dataclass
class Cfg:
    d: int = 512
    d_rec: int = 512
    n_layers: int = 6
    n_heads: int = 8
    mlp_mult: int = 4
    seq_len: int = 512
    lm_batch: int = 8
    rec_batch: int = 8
    lm_weight: float = 1.0
    lr: float = 6e-4
    warmup: int = 200
    weight_decay: float = 0.0
    grad_clip: float = 1.0
    n_keys: int = 256
    n_vals: int = 256
    n_query: int = 4
    filler: int = 64


CFG = Cfg()
if PRESET == "smoke":
    CFG.warmup, CFG.seq_len = 20, 256

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
IS_CUDA = DEVICE.type == "cuda"
GPU_NAME = torch.cuda.get_device_name(0) if IS_CUDA else "cpu"
AMP_DTYPE = torch.bfloat16 if (IS_CUDA and torch.cuda.get_device_capability()[0] >= 8) else torch.float16


def amp():
    return torch.autocast("cuda", dtype=AMP_DTYPE) if IS_CUDA else nullcontext()


def sync():
    if IS_CUDA:
        torch.cuda.synchronize()


def set_seed(s):
    torch.manual_seed(s); np.random.seed(s); random.seed(s)


TAG = f"d{CFG.d}_{PRESET}_v{FORMAT_VERSION}"
WORK = pathlib.Path(WORKDIR).expanduser()
RESDIR, CKPTDIR, FIGDIR, TABDIR, DATADIR = (
    WORK / f"{n}_{TAG}" for n in ("results", "checkpoints", "figures", "tables", "data"))
for _d in (RESDIR, CKPTDIR, FIGDIR, TABDIR, DATADIR):
    _d.mkdir(parents=True, exist_ok=True)
T_START = time.time()
T_END = T_START + TIME_BUDGET_H * 3600

print(f"{GPU_NAME} | {str(AMP_DTYPE).split('.')[-1]} | torch {torch.__version__}")
print(f"gate: retrieval-only at L={GATE_LOAD}, {LONG_STEPS} steps, viable at {VIABLE_ACC}")
print(f"work dir {WORK}")

In [ ]:
def keyof(spec):
    return hashlib.md5(json.dumps(spec, sort_keys=True, default=str).encode()).hexdigest()[:16]


class Records:
    def __init__(self, path):
        self.path, self.recs = path, {}
        if path.exists():
            for line in path.read_text().splitlines():
                try:
                    r = json.loads(line); self.recs[r["_key"]] = r
                except (json.JSONDecodeError, KeyError):
                    pass

    def get(self, spec):
        return self.recs.get(keyof(spec))

    def put(self, spec, payload):
        r = {"_key": keyof(spec), "_spec": spec, "_t": time.time(), **payload}
        self.recs[r["_key"]] = r
        with open(self.path, "a") as f:
            f.write(json.dumps(r, default=float) + "\n"); f.flush(); os.fsync(f.fileno())
        return r


def ckpt_save(name, obj):
    p = CKPTDIR / f"{name}.pt"
    torch.save(obj, p.with_suffix(".tmp")); os.replace(p.with_suffix(".tmp"), p)


def ckpt_load(name):
    p = CKPTDIR / f"{name}.pt"
    if p.exists():
        try:
            return torch.load(p, map_location=DEVICE, weights_only=False)
        except Exception as e:
            print(f"  checkpoint unreadable ({type(e).__name__})")
    return None


def ckpt_delete(name):
    (CKPTDIR / f"{name}.pt").unlink(missing_ok=True)


def write_once(path, obj):
    if not path.exists():
        path.write_text(json.dumps(obj, indent=1))
    return json.loads(path.read_text())


REC = Records(RESDIR / "records.jsonl")
print(f"{len(REC.recs)} finished run(s) on disk")

# The decision rule is recorded BEFORE any training, so the verdict cannot be chosen afterwards.
RULE = write_once(RESDIR / "gate2_rule.json", {
    "created": time.strftime("%Y-%m-%d %H:%M:%S %Z"),
    "question": "is retrieval-only at L=3 capacity-limited, or was the earlier 6000-step budget "
                "simply too short?",
    "previous_result": {"steps": 6000, "retrieval_only_acc": PREV_ACC},
    "load": GATE_LOAD, "steps": LONG_STEPS, "viable_acc": VIABLE_ACC,
    "still_rising": STILL_RISING,
    "verdicts": {
        "VIABLE": f"retrieval-only reaches {VIABLE_ACC} at any logged checkpoint",
        "CEILING": f"never reaches {VIABLE_ACC} and gains < {STILL_RISING} over the final third",
        "UNDERTRAINED": f"never reaches {VIABLE_ACC} but still gaining > {STILL_RISING}"},
    "followups": {"if VIABLE": "run the free hybrid at the same load and check retrieval dominance",
                  "otherwise": "report as a measured boundary; do not extend the budget again"}})
print(f"decision rule recorded {RULE['created']}: {RULE['verdicts']}")

In [ ]:
# WikiText-103 through the GPT-2 tokenizer, remapped to the most frequent LM_VOCAB tokens so the
# embedding and softmax stay small. Recall symbols are appended after the text vocabulary, so both
# objectives share one output layer. Cached: the first run pays the tokenisation cost once.
from transformers import AutoTokenizer


def prepare_text():
    stem = f"wikitext103_{N_STORIES}_v{LM_VOCAB}"
    names = [f"{stem}_{s}" for s in ("train.npy", "val.npy", "meta.json")]
    for root in SEARCH_ROOTS:                     # reuse a cache attached as a Kaggle dataset
        hits = [sorted(glob.glob(str(root / "**" / n), recursive=True)) for n in names]
        if all(hits):
            print(f"  reusing tokenised text from {pathlib.Path(hits[0][-1]).parent}")
            return (np.load(hits[0][-1], mmap_mode="r"), np.load(hits[1][-1], mmap_mode="r"),
                    json.loads(pathlib.Path(hits[2][-1]).read_text()))
    ftr, fva, fmeta = (DATADIR / n for n in names)
    from datasets import load_dataset
    tok = AutoTokenizer.from_pretrained("gpt2")
    try:
        ds = load_dataset("Salesforce/wikitext", "wikitext-103-raw-v1")
    except Exception:
        ds = load_dataset("wikitext", "wikitext-103-raw-v1")

    def enc(split, limit=None):
        txt = [t for t in ds[split]["text"] if len(t.strip()) > 40]
        if limit:
            txt = txt[:limit]
        out, t0 = [], time.time()
        for i in range(0, len(txt), 2000):
            out.append(np.asarray(sum(tok(txt[i:i + 2000], add_special_tokens=False)["input_ids"], []),
                                  dtype=np.int32))
            if (i // 2000) % 20 == 0 and i:
                print(f"  {split}: {i:,} paragraphs, {(time.time() - t0) / 60:.1f} min", flush=True)
        return np.concatenate(out)

    raw_tr, raw_va = enc("train", N_STORIES), enc("validation")
    counts = np.bincount(raw_tr.astype(np.int64), minlength=len(tok))
    keep = np.argsort(-counts, kind="stable")[: LM_VOCAB - 1]
    lut = np.zeros(len(tok), dtype=np.uint16)
    lut[keep] = np.arange(1, LM_VOCAB, dtype=np.uint16)
    np.save(ftr, lut[raw_tr]); np.save(fva, lut[raw_va])
    meta = {"train_tokens": int(len(raw_tr)), "val_tokens": int(len(raw_va)),
            "coverage": float(counts[keep].sum() / counts.sum())}
    fmeta.write_text(json.dumps(meta))
    return np.load(ftr, mmap_mode="r"), np.load(fva, mmap_mode="r"), meta


TRAIN, VAL, META = prepare_text()
UNK = 0
SEP, QRY = LM_VOCAB, LM_VOCAB + 1
KEY0 = LM_VOCAB + 2
VAL0 = KEY0 + CFG.n_keys
VOCAB = VAL0 + CFG.n_vals
IGNORE = -100
CHANCE = 1.0 / CFG.n_vals
print(f"text: {META['train_tokens']:,} train tokens, {META['val_tokens']:,} val; "
      f"top-{LM_VOCAB - 1} covers {100 * META['coverage']:.2f}% | full vocab {VOCAB}")


def lm_batch(B, data, rng):
    T = CFG.seq_len
    ix = rng.integers(0, len(data) - T - 1, size=B)
    arr = np.stack([np.asarray(data[i:i + T + 1], dtype=np.int64) for i in ix])
    t = torch.from_numpy(arr).to(DEVICE, non_blocking=True)
    return t[:, :-1].contiguous(), t[:, 1:].contiguous()


def recall_batch(B, load, gen, data, rng):
    dev = gen.device
    nq = min(CFG.n_query, load)
    T = 1 + load + CFG.filler + 1 + 2 * nq
    keys = torch.rand(B, CFG.n_keys, generator=gen, device=dev).argsort(-1)[:, :load]
    vals = torch.randint(0, CFG.n_vals, (B, load), generator=gen, device=dev)
    qidx = torch.rand(B, load, generator=gen, device=dev).argsort(-1)[:, :nq]
    x_key = torch.full((B, T), UNK, dtype=torch.long, device=dev)
    x_val = torch.full((B, T), UNK, dtype=torch.long, device=dev)
    y = torch.full((B, T), IGNORE, dtype=torch.long, device=dev)
    x_key[:, 0] = SEP
    x_key[:, 1:1 + load] = KEY0 + keys
    x_val[:, 1:1 + load] = VAL0 + vals
    p = 1 + load
    ix = rng.integers(0, len(data) - CFG.filler - 1, size=B)
    filler = torch.from_numpy(np.stack([np.asarray(data[i:i + CFG.filler], dtype=np.int64) for i in ix]))
    x_key[:, p:p + CFG.filler] = filler.to(dev)
    p += CFG.filler
    x_key[:, p] = SEP
    qpos = p + 1 + 2 * torch.arange(nq, device=dev)
    x_key[:, qpos] = QRY
    x_key[:, qpos + 1] = KEY0 + keys.gather(1, qidx)
    y[:, qpos + 1] = VAL0 + vals.gather(1, qidx)
    return (x_key.to(DEVICE, non_blocking=True), x_val.to(DEVICE, non_blocking=True),
            y.to(DEVICE, non_blocking=True))


def eval_sets(load):
    g = torch.Generator(device="cpu"); g.manual_seed(EVAL_SEED + load)
    rng = np.random.default_rng(EVAL_SEED + load)
    rec = [recall_batch(32, load, g, VAL, rng) for _ in range(N_EVAL_REC)]
    rng2 = np.random.default_rng(EVAL_SEED)
    lm = [lm_batch(8, VAL, rng2) for _ in range(N_EVAL_LM)]
    return rec, lm


print(f"recall probe at L={GATE_LOAD}: {min(CFG.n_query, GATE_LOAD)} queries per sequence, "
      f"chance {CHANCE:.4f}")

In [ ]:
# The patched dual-route readout. BOTH routes read the trunk states S; retrieval no longer reads
# the GRU output, so deleting or pricing one route leaves the other intact. This is the only
# substantive change from the earlier pilot.
class RMSNorm(nn.Module):
    def __init__(self, d, eps=1e-6):
        super().__init__()
        self.w, self.eps = nn.Parameter(torch.ones(d)), eps

    def forward(self, x):
        return self.w * x * torch.rsqrt(x.float().pow(2).mean(-1, keepdim=True) + self.eps).to(x.dtype)


def rope(T, dh, device, base=10000.0):
    inv = 1.0 / (base ** (torch.arange(0, dh, 2, device=device).float() / dh))
    f = torch.outer(torch.arange(T, device=device).float(), inv)
    return f.cos(), f.sin()


def apply_rope(x, cos, sin):
    x1, x2 = x[..., ::2], x[..., 1::2]
    cos, sin = cos[None, None].to(x.dtype), sin[None, None].to(x.dtype)
    return torch.stack([x1 * cos - x2 * sin, x1 * sin + x2 * cos], -1).flatten(-2)


class Block(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.h, self.dh = cfg.n_heads, cfg.d // cfg.n_heads
        self.n1, self.n2 = RMSNorm(cfg.d), RMSNorm(cfg.d)
        self.qkv = nn.Linear(cfg.d, 3 * cfg.d, bias=False)
        self.o = nn.Linear(cfg.d, cfg.d, bias=False)
        self.mlp = nn.Sequential(nn.Linear(cfg.d, cfg.mlp_mult * cfg.d, bias=False), nn.GELU(),
                                 nn.Linear(cfg.mlp_mult * cfg.d, cfg.d, bias=False))

    def forward(self, x, cos, sin):
        B, T, D = x.shape
        q, k, v = self.qkv(self.n1(x)).view(B, T, 3, self.h, self.dh).permute(2, 0, 3, 1, 4)
        q, k = apply_rope(q, cos, sin), apply_rope(k, cos, sin)
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        x = x + self.o(y.transpose(1, 2).reshape(B, T, D))
        return x + self.mlp(self.n2(x))


class DualRouteLM(nn.Module):
    def __init__(self, cfg, recurrent=True, retrieval=True):
        super().__init__()
        self.cfg, self.recurrent, self.retrieval = cfg, recurrent, retrieval
        self.emb_key = nn.Embedding(VOCAB, cfg.d)
        self.emb_val = nn.Embedding(VOCAB, cfg.d)
        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layers)])
        self.norm = RMSNorm(cfg.d)
        self.lm_head = nn.Linear(cfg.d, VOCAB, bias=False)
        if recurrent:
            self.gru = nn.GRU(cfg.d, cfg.d_rec, batch_first=True)
            self.proj_h = nn.Linear(cfg.d_rec, cfg.d_rec)
            self.w_h = nn.Linear(cfg.d_rec, VOCAB, bias=False)
        if retrieval:
            # PATCHED: q, k and v read the trunk width, never the GRU width, because they now read S
            self.q = nn.Linear(cfg.d, cfg.d, bias=False)
            self.k = nn.Linear(cfg.d, cfg.d, bias=False)
            self.v = nn.Linear(cfg.d, cfg.d, bias=False)
            self.proj_c = nn.Linear(cfg.d, cfg.d)
            self.w_c = nn.Linear(cfg.d, VOCAB, bias=False)
        for m in self.modules():
            if isinstance(m, (nn.Linear, nn.Embedding)):
                nn.init.normal_(m.weight, std=0.02)
            if isinstance(m, nn.Linear) and m.bias is not None:
                nn.init.zeros_(m.bias)

    def trunk(self, x_key, x_val):
        h = self.emb_key(x_key) + self.emb_val(x_val)
        cos, sin = rope(h.shape[1], self.cfg.d // self.cfg.n_heads, h.device)
        for blk in self.blocks:
            h = blk(h, cos, sin)
        return self.norm(h)

    def lm_logits(self, x_key, x_val):
        return self.lm_head(self.trunk(x_key, x_val))

    def routes(self, S, qpos):
        B, T, _ = S.shape
        h_rec = None
        if self.recurrent:
            G = self.gru(S)[0]
            h_rec = G.gather(1, qpos[None, :, None].expand(B, -1, G.shape[-1]))
        c = None
        if self.retrieval:
            # PATCHED: query, keys and values all come from S, so retrieval does not depend on
            # the recurrent route. The two are parallel given the trunk.
            idx_s = qpos[None, :, None].expand(B, -1, S.shape[-1])
            q = self.q(S.gather(1, idx_s))
            k, v = self.k(S), self.v(S)
            pos = torch.arange(T, device=S.device)
            allowed = pos[None, :] < qpos[:, None]          # strictly earlier: no self-leak
            mask = torch.zeros(len(qpos), T, device=S.device).masked_fill(~allowed, float("-inf"))
            c = F.scaled_dot_product_attention(q.unsqueeze(1), k.unsqueeze(1), v.unsqueeze(1),
                                               attn_mask=mask[None, None]).squeeze(1)
        return h_rec, c

    def recall_logits(self, x_key, x_val, qpos, drop=None, perm=None):
        S = self.trunk(x_key, x_val)
        h_rec, c = self.routes(S, qpos)
        ref = h_rec if h_rec is not None else c
        if drop == "recurrent":
            h_rec = None
        elif perm is not None and perm[0] == "recurrent" and h_rec is not None:
            h_rec = h_rec[perm[1]]
        if drop == "retrieval":
            c = None
        elif perm is not None and perm[0] == "retrieval" and c is not None:
            c = c[perm[1]]
        lh = self.w_h(F.gelu(self.proj_h(h_rec))) if (self.recurrent and h_rec is not None) else None
        lc = self.w_c(F.gelu(self.proj_c(c))) if (self.retrieval and c is not None) else None
        if lh is None and lc is None:
            return torch.zeros(ref.shape[0], ref.shape[1], VOCAB, device=ref.device), None, None
        out = lh if lc is None else (lc if lh is None else lh + lc)
        return out, lh, lc


def build(kind, seed):
    set_seed(seed)
    return DualRouteLM(CFG, recurrent=kind != "retrieval_only",
                       retrieval=kind != "recurrent_only").to(DEVICE)


# independence self-test: with the patch, deleting recurrence must not change the retrieval logits
_m = build("hybrid", 0)
_g = torch.Generator(device=DEVICE); _g.manual_seed(0)
_k_, _v_, _y_ = recall_batch(4, GATE_LOAD, _g, VAL, np.random.default_rng(0))
_qp = (_y_[0] != IGNORE).nonzero(as_tuple=True)[0]
with torch.no_grad():
    _lc_full = _m.recall_logits(_k_, _v_, _qp)[2]
    _lc_norec = _m.recall_logits(_k_, _v_, _qp, drop="recurrent")[2]
_gap = float((_lc_full - _lc_norec).abs().max())
print(f"route independence check: max |l_c with recurrence - l_c without| = {_gap:.2e}")
assert _gap < 1e-4, ("retrieval still depends on the recurrent route; the parallel-route patch is "
                     "not in effect")
for _kind in ("recurrent_only", "retrieval_only", "hybrid"):
    _mm = build(_kind, 0)
    _n = sum(p.numel() for p in _mm.parameters())
    print(f"{_kind:<16}{_n / 1e6:6.2f}M parameters")
del _m, _mm

In [ ]:
def qpos_of(y):
    return (y[0] != IGNORE).nonzero(as_tuple=True)[0]


@torch.no_grad()
def evaluate(model, rec_batches, lm_batches):
    was = model.training
    model.eval()
    tot = dict.fromkeys(["clean", "rec_res", "ret_res", "no_rec", "no_ret"], 0.0)
    n = 0
    for x_key, x_val, y in rec_batches:
        qpos = qpos_of(y)
        tgt = y[:, qpos]
        perm = torch.roll(torch.arange(x_key.shape[0], device=DEVICE), 1)

        def hit(**kw):
            with amp():
                lg = model.recall_logits(x_key, x_val, qpos, **kw)[0]
            return float((lg.argmax(-1) == tgt).sum())

        tot["clean"] += hit()
        if model.recurrent:
            tot["rec_res"] += hit(perm=("recurrent", perm)); tot["no_rec"] += hit(drop="recurrent")
        if model.retrieval:
            tot["ret_res"] += hit(perm=("retrieval", perm)); tot["no_ret"] += hit(drop="retrieval")
        n += tgt.numel()
    m = {k: v / n for k, v in tot.items()}
    acc = m["clean"]
    head = max(acc - CHANCE, 1e-9)
    nll, ntok = 0.0, 0
    for x, y in lm_batches:
        with amp():
            lg = model.lm_logits(x, torch.zeros_like(x))
        nll += float(F.cross_entropy(lg.float().reshape(-1, VOCAB), y.reshape(-1), reduction="sum"))
        ntok += y.numel()
    if was:
        model.train()
    return {"acc": acc, "ppl": math.exp(nll / ntok),
            "D_rec": float(np.clip((acc - m["rec_res"]) / head, 0, 1)) if model.recurrent else float("nan"),
            "D_ret": float(np.clip((acc - m["ret_res"]) / head, 0, 1)) if model.retrieval else float("nan"),
            "acc_no_rec": m["no_rec"] if model.recurrent else acc,
            "acc_no_ret": m["no_ret"] if model.retrieval else acc}


def train_run(key, kind, load, seed, steps):
    model = build(kind, seed)
    opt = torch.optim.AdamW(model.parameters(), lr=CFG.lr, betas=(0.9, 0.95),
                            weight_decay=CFG.weight_decay)
    scaler = torch.amp.GradScaler("cuda", enabled=IS_CUDA and AMP_DTYPE == torch.float16)
    rec_b, lm_b = eval_sets(load)
    start, trace, rt, rs = 0, [], 0.0, 0
    ck = ckpt_load(f"run_{key}")
    if ck is not None:
        model.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"])
        if ck["scaler"]:
            scaler.load_state_dict(ck["scaler"])
        start, trace, rt, rs = ck["step"], ck["trace"], ck["rt"], ck["rs"]
        print(f"    resumed at step {start}/{steps}", flush=True)
    rng = np.random.default_rng([seed, int(key[:8], 16), start])
    gen = torch.Generator(device=DEVICE); gen.manual_seed(int(rng.integers(2 ** 62)))
    model.train()
    last_ck, lm_l, rc_l = time.time(), [], []
    for step in range(start + 1, steps + 1):
        sync(); t0 = time.time()
        for g in opt.param_groups:
            g["lr"] = CFG.lr * min(1.0, step / CFG.warmup)
        opt.zero_grad(set_to_none=True)
        x, y = lm_batch(CFG.lm_batch, TRAIN, rng)
        with amp():
            lg = model.lm_logits(x, torch.zeros_like(x))
        loss_lm = F.cross_entropy(lg.float().reshape(-1, VOCAB), y.reshape(-1))
        scaler.scale(CFG.lm_weight * loss_lm).backward()
        x_key, x_val, yr = recall_batch(CFG.rec_batch, load, gen, TRAIN, rng)
        qpos = qpos_of(yr)
        with amp():
            out = model.recall_logits(x_key, x_val, qpos)[0]
        loss_rc = F.cross_entropy(out.float().reshape(-1, VOCAB), yr[:, qpos].reshape(-1))
        scaler.scale(loss_rc).backward()
        scaler.unscale_(opt)
        nn.utils.clip_grad_norm_(model.parameters(), CFG.grad_clip)
        scaler.step(opt); scaler.update()
        sync(); rt += time.time() - t0; rs += 1
        lm_l.append(float(loss_lm.detach())); rc_l.append(float(loss_rc.detach()))
        if step == start + 20:
            print(f"    {1000 * rt / rs:.0f} ms/step, about {rt / rs * (steps - step) / 60:.0f} min left",
                  flush=True)
        if step % LOG == 0 or step == steps:
            ev = evaluate(model, rec_b, lm_b)
            trace.append({"step": step, "lm_loss": float(np.mean(lm_l)),
                          "rc_loss": float(np.mean(rc_l)), **ev})
            best = max(t["acc"] for t in trace)
            print(f"    step {step:>6}  lm {np.mean(lm_l):.3f} (ppl {ev['ppl']:.1f})  "
                  f"recall {np.mean(rc_l):.3f}  acc {ev['acc']:.3f} (best {best:.3f})"
                  + (f"  D_rec {ev['D_rec']:.2f} D_ret {ev['D_ret']:.2f}" if kind == "hybrid" else ""),
                  flush=True)
            lm_l, rc_l = [], []
        if step < steps and time.time() - last_ck > 60 * CKPT_MINUTES:
            ckpt_save(f"run_{key}", {"model": model.state_dict(), "opt": opt.state_dict(),
                                     "scaler": scaler.state_dict(), "step": step, "trace": trace,
                                     "rt": rt, "rs": rs})
            last_ck = time.time()
        if time.time() > T_END:
            ckpt_save(f"run_{key}", {"model": model.state_dict(), "opt": opt.state_dict(),
                                     "scaler": scaler.state_dict(), "step": step, "trace": trace,
                                     "rt": rt, "rs": rs})
            print(f"    time budget reached at step {step}; checkpoint saved", flush=True)
            del model, opt
            if IS_CUDA:
                torch.cuda.empty_cache()
            return None
    out = {**evaluate(model, rec_b, lm_b), "trace": trace, "ms_per_step": 1000 * rt / max(rs, 1),
           "params": sum(p.numel() for p in model.parameters()), "steps_done": steps}
    ckpt_delete(f"run_{key}")
    del model, opt
    if IS_CUDA:
        torch.cuda.empty_cache()
    return out


def verdict(trace):
    """Applies the rule recorded in gate2_rule.json. Returns (verdict, best, gain over final third)."""
    accs = [t["acc"] for t in trace]
    best = max(accs)
    third = max(1, len(accs) // 3)
    gain = float(np.mean(accs[-third:]) - np.mean(accs[-2 * third:-third])) if len(accs) >= 2 * third else float("nan")
    if best >= VIABLE_ACC:
        return "VIABLE", best, gain
    if gain == gain and gain < STILL_RISING:
        return "CEILING", best, gain
    return "UNDERTRAINED", best, gain

In [ ]:
# Run 1 is the gate. Runs 2 and 3 only make sense if it passes, so they are conditional.
print(f">>> gate run: retrieval_only at L={GATE_LOAD} for {LONG_STEPS} steps", flush=True)
BASE = {"cfg": asdict(CFG), "format": FORMAT_VERSION, "load": GATE_LOAD}
G_SPEC = {**BASE, "block": "gate", "kind": "retrieval_only", "seed": SEEDS[0], "steps": LONG_STEPS}
GATE = REC.get(G_SPEC)
if GATE is None:
    res = train_run(keyof(G_SPEC), "retrieval_only", GATE_LOAD, SEEDS[0], LONG_STEPS)
    if res is None:
        raise RuntimeError("the gate run did not finish in the time budget. Its checkpoint is "
                           "saved: Save Version, attach this output as a dataset, run all again.")
    GATE = REC.put(G_SPEC, res)
else:
    print("  gate run finished earlier", flush=True)

GT = pd.DataFrame(GATE["trace"])
V, BEST, GAIN = verdict(GATE["trace"])
print(f"\n{'=' * 74}")
print(f"VERDICT: {V}")
print(f"  best retrieval-only accuracy {BEST:.4f} (earlier 6000-step run: {PREV_ACC:.4f})")
print(f"  change over the final third: {GAIN:+.4f} (rule threshold {STILL_RISING})")
print(f"  language modelling perplexity at the end: {GT.ppl.iloc[-1]:.1f}")
print(f"{'=' * 74}\n", flush=True)

FOLLOWUP = []
if V == "VIABLE":
    print("retrieval is a viable receiver. Running the free hybrid to check whether it is "
          "retrieval-dominant, which is the precondition for steering.", flush=True)
    FOLLOWUP = [("hybrid", LONG_STEPS // 2), ("recurrent_only", LONG_STEPS // 2)]
else:
    print("retrieval is not viable at this load, so steering is not testable here. The follow-up "
          "runs are skipped: this is a measured boundary, and extending the budget again would "
          "make the screen post-hoc.", flush=True)

RESULTS = {"retrieval_only": GATE}
for kind, steps in FOLLOWUP:
    if time.time() > T_END:
        print(f"  skipping {kind}: time budget spent", flush=True)
        continue
    spec = {**BASE, "block": "followup", "kind": kind, "seed": SEEDS[0], "steps": steps}
    got = REC.get(spec)
    if got is None:
        print(f">>> {kind} at L={GATE_LOAD} for {steps} steps", flush=True)
        got = train_run(keyof(spec), kind, GATE_LOAD, SEEDS[0], steps)
        if got is None:
            print(f"  {kind} paused at the budget; run all again to continue", flush=True)
            continue
        got = REC.put(spec, got)
    RESULTS[kind] = got
    print(f"  {kind}: acc {got['acc']:.3f}, ppl {got['ppl']:.1f}"
          + (f", D_rec {got['D_rec']:.2f}, D_ret {got['D_ret']:.2f}, "
             f"no-ret {got['acc_no_ret']:.3f}, no-rec {got['acc_no_rec']:.3f}"
             if kind == "hybrid" else ""), flush=True)

In [ ]:
lines = [f"=== PARALLEL-ROUTE LM PILOT, GATE 2 | {time.strftime('%Y-%m-%d %H:%M')} ===",
         f"{GPU_NAME} | trunk d={CFG.d} x {CFG.n_layers} layers | GRU d_rec={CFG.d_rec} | "
         f"filler {CFG.filler} tokens | chance {CHANCE:.4f}",
         f"routes read the trunk states S (patched); independence check passed at {_gap:.1e}",
         f"decision rule recorded {RULE['created']} before training",
         "",
         f"QUESTION: at L={GATE_LOAD}, was retrieval-only's {PREV_ACC:.4f} after 6000 steps a "
         f"capacity ceiling or too small a budget?",
         f"ANSWER:   {V} - best {BEST:.4f} over {LONG_STEPS} steps, "
         f"final-third change {GAIN:+.4f}",
         "",
         "-- retrieval-only accuracy against training step --",
         GT[["step", "acc", "ppl", "rc_loss"]].round(4).to_string(index=False),
         ""]

if "hybrid" in RESULTS:
    h = RESULTS["hybrid"]
    dom = h["D_ret"] > h["D_rec"]
    lines += ["-- free hybrid at the same load --",
              f"  accuracy {h['acc']:.3f} | D_ret {h['D_ret']:.2f} | D_rec {h['D_rec']:.2f}",
              f"  deleting retrieval leaves {h['acc_no_ret']:.3f}, deleting recurrence leaves "
              f"{h['acc_no_rec']:.3f}",
              f"  free hybrid is {'RETRIEVAL' if dom else 'RECURRENCE'}-dominant"
              + ("  -> steering is testable here; the four-arm run is the next session"
                 if dom else "  -> the free hybrid already prefers recurrence, so there is no "
                             "retrieval-dominant solution to steer away from"), ""]
if "recurrent_only" in RESULTS:
    r = RESULTS["recurrent_only"]
    lines += [f"-- recurrent-only reference: acc {r['acc']:.3f}, ppl {r['ppl']:.1f}", ""]

lines += ["INTERPRETATION",
          {"VIABLE": "  Retrieval can carry the task at this load, so the earlier screen was "
                     "budget-limited rather than capacity-limited. Report the corrected screen.",
           "CEILING": "  Retrieval plateaus below the threshold: a measured capacity limit, not an "
                      "artifact of the budget. This is a boundary row - the route cannot be the "
                      "receiver here. Do not extend the budget again.",
           "UNDERTRAINED": "  Retrieval is still improving at the end of training, so neither "
                           "conclusion is available. Report the budget as insufficient rather "
                           "than claiming a ceiling."}[V]]

mpl.rcParams.update({"figure.dpi": 130, "savefig.dpi": 300, "savefig.bbox": "tight", "font.size": 8,
                     "axes.spines.top": False, "axes.spines.right": False})
fig, ax = plt.subplots(1, 2, figsize=(7.0, 2.5))
ax[0].plot(GT.step, GT.acc, "-o", ms=2.5, label="retrieval only")
for kind, style in (("hybrid", "-s"), ("recurrent_only", "-^")):
    if kind in RESULTS:
        t = pd.DataFrame(RESULTS[kind]["trace"])
        ax[0].plot(t.step, t.acc, style, ms=2.5, label=kind.replace("_", " "))
ax[0].axhline(VIABLE_ACC, color="0.6", lw=0.8, ls="--")
ax[0].axhline(PREV_ACC, color="0.8", lw=0.8, ls=":")
ax[0].set_xlabel("training step"); ax[0].set_ylabel("recall accuracy")
ax[0].set_title(f"L={GATE_LOAD}: is retrieval capacity-limited?"); ax[0].legend(frameon=False)
ax[1].plot(GT.step, GT.ppl, "-o", ms=2.5, color="tab:red")
ax[1].set_xlabel("training step"); ax[1].set_ylabel("validation perplexity")
ax[1].set_title("language modelling is preserved")
fig.tight_layout()
for ext in ("pdf", "png"):
    fig.savefig(FIGDIR / f"gate2.{ext}")
plt.show()

SUMMARY = "\n".join(lines)
(TABDIR / "GATE2_SUMMARY.txt").write_text(SUMMARY)
GT.to_csv(TABDIR / "gate2_retrieval_only.csv", index=False)
print(SUMMARY)
print(f"\nsaved {TABDIR / 'GATE2_SUMMARY.txt'}")